# Evaluate Self-Play Data

This notebook inspects saved self-play outputs for the maintained adversarial pipelines.

Supported formats:
- `flat_meta` from `AlphaZeroMetaAdversarial`
- `factorized_axes` from `AlphaZeroAdversarial`

It focuses on the data-quality aspects that matter before training:
- manifest and shard integrity
- outcome balance and episode length
- sample balance by role
- policy concentration and target-value distribution
- action bias for both flat-meta and factorized policy formats
- target-vector sanity
- spot checks of saved tensors


In [ ]:
from __future__ import annotations

import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch

try:
    from IPython.display import display
except ModuleNotFoundError:
    def display(value):
        print(value)

plt.style.use("ggplot")
pd.set_option("display.max_columns", 160)
pd.set_option("display.width", 200)

ROOT = Path.cwd().resolve()
if not (ROOT / "source").exists():
    ROOT = Path("/home/fannam/workspace/Autonomous-Driving-Gym").resolve()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from notebooks.analysis.evaluate_self_play_data_support import (
    META_ACTION_LABELS,
    analyze_manifest,
    resolve_manifest_path,
    summarize_manifest,
)

SEARCH_ROOTS = [
    ROOT / "source/AlphaZero-meta-adversarial-autonomous-driving/outputs/self_play",
    ROOT / "source/AlphaZero-adversarial-autonomous-driving/outputs/self_play",
    ROOT / "outputs",
    Path("/kaggle/working/alphazero_meta_adversarial_self_play"),
    Path("/kaggle/working/alphazero_adversarial_self_play"),
]
MANIFEST_PATH: str | None = None
STATE_SAMPLE_INDEX = 0

ROOT


In [ ]:
manifest_path = resolve_manifest_path(MANIFEST_PATH, SEARCH_ROOTS)
analysis = analyze_manifest(manifest_path)

manifest = analysis["manifest"]
policy_format = analysis["policy_format"]
factorized_axis_info = analysis["factorized_axis_info"]
shard_df = analysis["shard_df"]
episode_df = analysis["episode_df"]
sample_df = analysis["sample_df"]
target_stats_df = analysis["target_stats_df"]

display(summarize_manifest(analysis))
display(pd.DataFrame([manifest.get("args", {})]).T.rename(columns={0: "value"}).head(30))
display(shard_df.head(10))
display(episode_df.head(10))
print(f"policy_format={policy_format}")


In [ ]:
outcome_counts = (
    episode_df["outcome_reason"]
    .value_counts(dropna=False)
    .rename_axis("outcome_reason")
    .reset_index(name="episodes")
)
policy_mode_counts = (
    episode_df["policy_modes_label"]
    .value_counts(dropna=False)
    .rename_axis("policy_modes_label")
    .reset_index(name="episodes")
)
episode_numeric_cols = [
    column
    for column in ["steps", "sample_count", "ego_samples", "npc_samples", "elapsed_s", "ego_value", "npc_value"]
    if column in episode_df.columns
]

display(outcome_counts)
display(policy_mode_counts)
display(episode_df[episode_numeric_cols].describe().T)
display(
    episode_df.groupby("outcome_reason")[episode_numeric_cols]
    .agg(["count", "mean", "median"])
    .sort_values((episode_numeric_cols[0], "count"), ascending=False)
)

fig, axes = plt.subplots(2, 2, figsize=(18, 10))
outcome_counts.plot.bar(x="outcome_reason", y="episodes", ax=axes[0, 0], legend=False, color="tab:blue")
axes[0, 0].set_title("Episode outcomes")
axes[0, 0].set_xlabel("outcome_reason")
axes[0, 0].set_ylabel("episodes")

episode_df["steps"].plot.hist(
    bins=min(30, max(5, int(episode_df["steps"].nunique()))),
    ax=axes[0, 1],
    color="tab:orange",
)
axes[0, 1].set_title("Episode steps")
axes[0, 1].set_xlabel("steps")

episode_df["sample_count"].plot.hist(
    bins=min(30, max(5, int(episode_df["sample_count"].nunique()))),
    ax=axes[1, 0],
    color="tab:green",
)
axes[1, 0].set_title("Samples per episode")
axes[1, 0].set_xlabel("sample_count")

axes[1, 1].scatter(episode_df["steps"], episode_df["sample_count"], alpha=0.7, color="tab:red")
axes[1, 1].set_title("Steps vs samples")
axes[1, 1].set_xlabel("steps")
axes[1, 1].set_ylabel("sample_count")

plt.tight_layout()
plt.show()


In [ ]:
role_counts = (
    sample_df["role"]
    .value_counts(dropna=False)
    .rename_axis("role")
    .reset_index(name="samples")
)
sample_summary_cols = ["policy_entropy", "max_policy_prob", "value"]
for optional_column in ["accelerate_entropy", "steering_entropy"]:
    if optional_column in sample_df.columns:
        sample_summary_cols.append(optional_column)
sample_summary = (
    sample_df.groupby("role")[sample_summary_cols]
    .agg(["count", "mean", "median", "std"])
    .sort_index()
)

display(role_counts)
display(sample_summary)
display(target_stats_df)


In [ ]:
if policy_format == "flat_meta":
    action_distribution = pd.crosstab(
        sample_df["role"],
        sample_df["action_label"],
        normalize="index",
    ).reindex(columns=list(META_ACTION_LABELS.values()), fill_value=0.0)
    idle_rate = (
        sample_df.assign(is_idle=sample_df["action_label"].eq("IDLE"))
        .groupby("role")["is_idle"]
        .mean()
        .rename("idle_argmax_rate")
        .reset_index()
    )

    display(idle_rate)
    display(action_distribution)

    fig, axes = plt.subplots(2, 2, figsize=(18, 10))
    role_counts.plot.bar(x="role", y="samples", ax=axes[0, 0], legend=False, color="tab:purple")
    axes[0, 0].set_title("Sample balance by role")
    axes[0, 0].set_xlabel("role")
    axes[0, 0].set_ylabel("samples")

    action_distribution.T.plot.bar(ax=axes[0, 1])
    axes[0, 1].set_title("Argmax action distribution by role")
    axes[0, 1].set_xlabel("action")
    axes[0, 1].set_ylabel("fraction of samples")

    for role_name, frame in sample_df.groupby("role"):
        axes[1, 0].hist(frame["policy_entropy"], bins=40, alpha=0.6, label=role_name)
    axes[1, 0].set_title("Policy entropy by role")
    axes[1, 0].set_xlabel("entropy")
    axes[1, 0].legend()

    for role_name, frame in sample_df.groupby("role"):
        axes[1, 1].hist(frame["value"], bins=40, alpha=0.6, label=role_name)
    axes[1, 1].set_title("Value targets by role")
    axes[1, 1].set_xlabel("value")
    axes[1, 1].legend()

    plt.tight_layout()
    plt.show()

elif policy_format == "factorized_axes":
    axis_0_labels = factorized_axis_info["axis_0_labels"]
    axis_1_labels = factorized_axis_info["axis_1_labels"]
    accelerate_distribution = pd.crosstab(
        sample_df["role"],
        sample_df["accelerate_action_label"],
        normalize="index",
    ).reindex(columns=axis_0_labels, fill_value=0.0)
    steering_distribution = pd.crosstab(
        sample_df["role"],
        sample_df["steering_action_label"],
        normalize="index",
    ).reindex(columns=axis_1_labels, fill_value=0.0)
    center_joint_rate = (
        sample_df.groupby("role")["is_center_joint"]
        .mean()
        .rename("center_joint_rate")
        .reset_index()
    )
    top_joint_actions = (
        sample_df["action_label"]
        .value_counts(dropna=False)
        .rename_axis("joint_argmax_action")
        .reset_index(name="samples")
        .head(20)
    )

    display(center_joint_rate)
    display(accelerate_distribution)
    display(steering_distribution)
    display(top_joint_actions)

    fig, axes = plt.subplots(2, 2, figsize=(18, 10))
    role_counts.plot.bar(x="role", y="samples", ax=axes[0, 0], legend=False, color="tab:purple")
    axes[0, 0].set_title("Sample balance by role")
    axes[0, 0].set_xlabel("role")
    axes[0, 0].set_ylabel("samples")

    accelerate_distribution.T.plot.bar(ax=axes[0, 1])
    axes[0, 1].set_title("Argmax accelerate distribution by role")
    axes[0, 1].set_xlabel("accelerate bin")
    axes[0, 1].set_ylabel("fraction of samples")

    steering_distribution.T.plot.bar(ax=axes[1, 0])
    axes[1, 0].set_title("Argmax steering distribution by role")
    axes[1, 0].set_xlabel("steering bin")
    axes[1, 0].set_ylabel("fraction of samples")

    for role_name, frame in sample_df.groupby("role"):
        axes[1, 1].hist(frame["value"], bins=40, alpha=0.6, label=role_name)
    axes[1, 1].set_title("Value targets by role")
    axes[1, 1].set_xlabel("value")
    axes[1, 1].legend()

    plt.tight_layout()
    plt.show()

    heatmap_roles = ["all"] + [role_name for role_name in ["ego", "npc"] if role_name in set(sample_df["role"])]
    fig, axes = plt.subplots(1, len(heatmap_roles), figsize=(6 * len(heatmap_roles), 5), squeeze=False)
    color_image = None
    for axis, role_name in zip(axes[0], heatmap_roles):
        frame = sample_df if role_name == "all" else sample_df[sample_df["role"] == role_name]
        heatmap = pd.crosstab(
            frame["accelerate_action_label"],
            frame["steering_action_label"],
        ).reindex(index=axis_0_labels, columns=axis_1_labels, fill_value=0)
        color_image = axis.imshow(heatmap.to_numpy(), aspect="auto", cmap="viridis")
        axis.set_title(f"Joint argmax heatmap: {role_name}")
        axis.set_xlabel("steering bin")
        axis.set_ylabel("accelerate bin")
        axis.set_xticks(range(len(axis_1_labels)))
        axis.set_xticklabels(axis_1_labels, rotation=45, ha="right")
        axis.set_yticks(range(len(axis_0_labels)))
        axis.set_yticklabels(axis_0_labels)
    if color_image is not None:
        fig.colorbar(color_image, ax=axes.ravel().tolist(), shrink=0.8)
    plt.tight_layout()
    plt.show()

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    for role_name, frame in sample_df.groupby("role"):
        axes[0].hist(frame["policy_entropy"], bins=40, alpha=0.6, label=role_name)
        axes[1].hist(frame["accelerate_entropy"], bins=40, alpha=0.4, label=f"{role_name}:accelerate")
        axes[1].hist(frame["steering_entropy"], bins=40, alpha=0.4, label=f"{role_name}:steering")
    axes[0].set_title("Joint policy entropy by role")
    axes[0].set_xlabel("entropy")
    axes[0].legend()
    axes[1].set_title("Axis entropy by role")
    axes[1].set_xlabel("entropy")
    axes[1].legend(ncol=2, fontsize=8)
    plt.tight_layout()
    plt.show()

else:
    print(f"Unsupported or mixed policy format for plotting: {policy_format}")


In [ ]:
manifest_total_samples = int(manifest.get("total_samples", shard_df["sample_count"].sum()))
loaded_total_samples = int(shard_df["sample_count"].sum())
sample_count_matches = manifest_total_samples == loaded_total_samples
episode_sample_match = bool((episode_df["sample_count"] == episode_df["collected_samples"]).all()) if "collected_samples" in episode_df.columns else True
policy_rows_close = bool(np.allclose(sample_df["policy_row_sum"].to_numpy(), 1.0, atol=1e-4)) if not sample_df.empty else True
value_range_ok = bool(sample_df["value"].between(-1.0001, 1.0001).all()) if not sample_df.empty else True

print("Sanity checks")
print(f"  manifest total_samples matches loaded shards : {sample_count_matches}")
print(f"  episode sample_count matches collected_samples: {episode_sample_match}")
print(f"  policy rows sum to 1 within tolerance        : {policy_rows_close}")
print(f"  value targets stay in [-1, 1]                : {value_range_ok}")

display(shard_df)

first_shard_path = Path(shard_df.iloc[0]["path"])
payload = torch.load(first_shard_path, map_location="cpu")
states = payload["states"].detach().cpu().numpy()
target_vectors = payload["target_vectors"].detach().cpu().numpy()
values = payload["values"].detach().cpu().numpy().reshape(-1)

if len(states) == 0:
    print(f"No samples available in first shard: {first_shard_path}")
else:
    sample_index = int(np.clip(STATE_SAMPLE_INDEX, 0, len(states) - 1))
    sample_state = states[sample_index]
    sample_target = target_vectors[sample_index] if len(target_vectors) else np.array([])
    sample_value = values[sample_index] if len(values) else np.nan

    print(
        f"State viewer: shard={first_shard_path.name} sample_index={sample_index} "
        f"sample_state_shape={sample_state.shape}"
    )
    channel_limit = min(sample_state.shape[0], 12)
    cols = 4
    rows = int(np.ceil(channel_limit / cols))
    fig, axes = plt.subplots(rows, cols, figsize=(4 * cols, 3 * rows))
    axes = np.asarray(axes).reshape(-1)
    for channel_index in range(channel_limit):
        axes[channel_index].imshow(sample_state[channel_index], cmap="viridis")
        axes[channel_index].set_title(f"channel {channel_index}")
        axes[channel_index].axis("off")
    for axis in axes[channel_limit:]:
        axis.axis("off")
    plt.tight_layout()
    plt.show()

    print("target_vector:", sample_target)
    if policy_format == "flat_meta":
        sample_policy = payload["policies"].detach().cpu().numpy()[sample_index]
        print("policy:", sample_policy)
    elif policy_format == "factorized_axes":
        sample_accelerate_policy = payload["accelerate_policies"].detach().cpu().numpy()[sample_index]
        sample_steering_policy = payload["steering_policies"].detach().cpu().numpy()[sample_index]
        print("accelerate_policy:", sample_accelerate_policy)
        print("steering_policy:", sample_steering_policy)
    print("value:", sample_value)
